<a href="https://colab.research.google.com/github/eshan14git/football-qa-nlp/blob/eshan-dev/notebooks/06_football_qa_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - Clone the correct GitHub branch

!git clone -b eshan-dev https://github.com/eshan14git/football-qa-nlp.git

Cloning into 'football-qa-nlp'...
remote: Enumerating objects: 205, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 205 (delta 49), reused 46 (delta 21), pack-reused 112 (from 4)
Receiving objects: 100% (205/205), 38.21 MiB | 20.39 MiB/s, done.
Resolving deltas: 100% (81/81), done.


In [2]:
# Cell 2 - Import libraries

import os
import re
import joblib
import pandas as pd

project_path = "/content/football-qa-nlp"
data_path = os.path.join(project_path, "data")
rf_path = os.path.join(project_path, "models", "random_forest")

print("Setup complete.")

Setup complete.


In [3]:
# Cell 3 - Load Random Forest model and TF-IDF vectorizer

rf_model = joblib.load(
    os.path.join(rf_path, "trained_random_forest.pkl")
)

rf_vectorizer = joblib.load(
    os.path.join(rf_path, "random_forest_tfidf_vectorizer.pkl")
)

print("Random Forest model loaded.")
print("TF-IDF vectorizer loaded.")

Random Forest model loaded.
TF-IDF vectorizer loaded.


In [4]:
# Cell 4 - Load cleaned datasets

results_df = pd.read_csv(
    os.path.join(data_path, "results_clean.csv")
)

goalscorers_df = pd.read_csv(
    os.path.join(data_path, "goalscorers_clean.csv")
)

shootouts_df = pd.read_csv(
    os.path.join(data_path, "shootouts_clean.csv")
)

former_names_df = pd.read_csv(
    os.path.join(data_path, "former_names_clean.csv")
)

print("Datasets loaded successfully.")
print("Results:", results_df.shape)
print("Goalscorers:", goalscorers_df.shape)
print("Shootouts:", shootouts_df.shape)
print("Former names:", former_names_df.shape)

Datasets loaded successfully.
Results: (49485, 9)
Goalscorers: (47855, 8)
Shootouts: (682, 5)
Former names: (36, 4)


In [5]:
# Cell 5 - Basic intent prediction function

def predict_intent(question):
    question_vector = rf_vectorizer.transform([question])
    prediction = rf_model.predict(question_vector)[0]
    return prediction

test_question = "Who won the 2014 FIFA World Cup Final?"

print("Question:", test_question)
print("Predicted intent:", predict_intent(test_question))

Question: Who won the 2014 FIFA World Cup Final?
Predicted intent: match_winner


In [6]:
# Cell 6 - Prepare searchable team and tournament lists

all_teams = sorted(
    set(results_df["home_team"].dropna().tolist()) |
    set(results_df["away_team"].dropna().tolist())
)

all_tournaments = sorted(
    results_df["tournament"].dropna().unique().tolist()
)

print("Unique teams:", len(all_teams))
print("Unique tournaments:", len(all_tournaments))

Unique teams: 336
Unique tournaments: 200


In [7]:
# Cell 7 - Extract basic entities from a question

def extract_basic_entities(question):
    question_lower = question.lower()

    # Find teams mentioned in the question
    found_teams = [
        team for team in all_teams
        if team.lower() in question_lower
    ]

    # Find year
    year_match = re.search(r"\b(18|19|20)\d{2}\b", question)
    year = year_match.group() if year_match else None

    # Find tournament
    found_tournaments = [
        tournament for tournament in all_tournaments
        if tournament.lower() in question_lower
    ]

    return {
        "teams": found_teams,
        "year": year,
        "tournaments": found_tournaments
    }

In [8]:
# Cell 8 - Test entity extraction

test_questions = [
    "Who won between India and Myanmar?",
    "Who won between India and Myanmar in 2017?",
    "Who won the FIFA World Cup match between Germany and Argentina in 2014?",
    "What was the score between Brazil and Argentina in Copa América?"
]

for q in test_questions:
    print("\nQuestion:", q)
    print(extract_basic_entities(q))


Question: Who won between India and Myanmar?
{'teams': ['India', 'Myanmar'], 'year': None, 'tournaments': []}

Question: Who won between India and Myanmar in 2017?
{'teams': ['India', 'Myanmar'], 'year': '2017', 'tournaments': []}

Question: Who won the FIFA World Cup match between Germany and Argentina in 2014?
{'teams': ['Argentina', 'Germany'], 'year': '2014', 'tournaments': ['FIFA World Cup']}

Question: What was the score between Brazil and Argentina in Copa América?
{'teams': ['Argentina', 'Brazil'], 'year': None, 'tournaments': ['Copa América']}


In [9]:
# Cell 9 - Find candidate matches from extracted entities

def find_result_candidates(question):
    entities = extract_basic_entities(question)

    candidates = results_df.copy()

    teams = entities["teams"]
    year = entities["year"]
    tournaments = entities["tournaments"]

    # Filter by two detected teams
    if len(teams) >= 2:
        team1, team2 = teams[:2]

        candidates = candidates[
            (
                (candidates["home_team"] == team1) &
                (candidates["away_team"] == team2)
            )
            |
            (
                (candidates["home_team"] == team2) &
                (candidates["away_team"] == team1)
            )
        ]

    # Filter by year
    if year:
        candidates = candidates[
            candidates["date"].astype(str).str.startswith(year)
        ]

    # Filter by tournament
    if tournaments:
        tournament = tournaments[0]

        candidates = candidates[
            candidates["tournament"].str.lower() == tournament.lower()
        ]

    return candidates

In [10]:
# Cell 10 - Test candidate retrieval

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

print("Candidate matches found:", len(candidates))

display(
    candidates[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Candidate matches found: 2


,date,home_team,away_team,home_score,away_score,tournament,city,country
40535,2017-03-28,Myanmar,India,0,1,AFC Asian Cup qualification,Yangon,Myanmar
41204,2017-11-14,India,Myanmar,2,2,AFC Asian Cup qualification,Margao,India


In [11]:
# Cell 11 - Inspect differences between candidate matches

def inspect_candidate_differences(candidates):
    if len(candidates) <= 1:
        return {}

    differences = {}

    fields = [
        "date",
        "tournament",
        "city",
        "country",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "neutral"
    ]

    for field in fields:
        unique_values = candidates[field].dropna().astype(str).unique().tolist()

        if len(unique_values) > 1:
            differences[field] = unique_values

    return differences

In [12]:
# Cell 12 - Test ambiguity inspection

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

differences = inspect_candidate_differences(candidates)

print("Candidate matches:", len(candidates))
print("\nDifferences:")

for field, values in differences.items():
    print(f"- {field}: {values}")

Candidate matches: 2

Differences:
- date: ['2017-03-28', '2017-11-14']
- city: ['Yangon', 'Margao']
- country: ['Myanmar', 'India']
- home_team: ['Myanmar', 'India']
- away_team: ['India', 'Myanmar']
- home_score: ['0', '2']
- away_score: ['1', '2']


In [13]:
# Cell 13 - Generate a clarification question

def generate_clarification_question(candidates):
    if len(candidates) == 0:
        return "I couldn't find a matching game."

    if len(candidates) == 1:
        return None

    differences = inspect_candidate_differences(candidates)

    # Prefer tournament when tournaments differ
    if "tournament" in differences:
        options = differences["tournament"]
        return (
            "I found multiple matching games. "
            "Do you remember the tournament? "
            + "Options: "
            + ", ".join(options)
        )

    # Then country/location
    if "country" in differences:
        options = differences["country"]
        return (
            "I found multiple matching games. "
            "Do you remember which country the game was played in? "
            + "Options: "
            + ", ".join(options)
        )

    # Then city
    if "city" in differences:
        options = differences["city"]
        return (
            "I found multiple matching games. "
            "Do you remember the city? "
            + "Options: "
            + ", ".join(options)
        )

    # Then home team
    if "home_team" in differences:
        options = differences["home_team"]
        return (
            "I found multiple matching games. "
            "Do you remember which team was the home team? "
            + "Options: "
            + ", ".join(options)
        )

    # Then exact date only as a last resort
    if "date" in differences:
        options = differences["date"]
        return (
            "I found multiple matching games. "
            "Do you remember which date it was? "
            + "Options: "
            + ", ".join(options)
        )

    return (
        "I found multiple matching games, but I need one more detail "
        "to identify the correct one."
    )

In [14]:
# Cell 14 - Test clarification generation

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

clarification = generate_clarification_question(candidates)

print("Question:", question)
print("\nAssistant:", clarification)

Question: Who won between India and Myanmar in 2017?

Assistant: I found multiple matching games. Do you remember which country the game was played in? Options: Myanmar, India


In [15]:
question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

print("Candidate matches:", len(candidates))
print(generate_clarification_question(candidates))

Candidate matches: 2
I found multiple matching games. Do you remember which country the game was played in? Options: Myanmar, India


In [16]:
# Cell 15 - Apply a clarification reply to existing candidates

def apply_clarification(candidates, reply):
    reply_lower = reply.lower().strip()

    filtered = candidates.copy()

    # Year
    year_match = re.search(r"\b(18|19|20)\d{2}\b", reply)
    if year_match:
        year = year_match.group()
        filtered = filtered[
            filtered["date"].astype(str).str.startswith(year)
        ]

    # Tournament
    tournament_matches = [
        tournament
        for tournament in filtered["tournament"].dropna().unique()
        if tournament.lower() in reply_lower
    ]

    if tournament_matches:
        tournament = tournament_matches[0]
        filtered = filtered[
            filtered["tournament"].str.lower() == tournament.lower()
        ]

    # Country
    country_matches = [
        country
        for country in filtered["country"].dropna().unique()
        if str(country).lower() in reply_lower
    ]

    if country_matches:
        country = country_matches[0]
        filtered = filtered[
            filtered["country"].astype(str).str.lower() == str(country).lower()
        ]

    # City
    city_matches = [
        city
        for city in filtered["city"].dropna().unique()
        if str(city).lower() in reply_lower
    ]

    if city_matches:
        city = city_matches[0]
        filtered = filtered[
            filtered["city"].astype(str).str.lower() == str(city).lower()
        ]

    # Home team
    home_team_matches = [
        team
        for team in filtered["home_team"].dropna().unique()
        if str(team).lower() in reply_lower
    ]

    if home_team_matches:
        team = home_team_matches[0]
        filtered = filtered[
            filtered["home_team"].astype(str).str.lower() == str(team).lower()
        ]

    return filtered

In [17]:
# Cell 16 - Test clarification filtering

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

print("Initial candidates:", len(candidates))

clarification_reply = "India"

filtered_candidates = apply_clarification(
    candidates,
    clarification_reply
)

print("Candidates after clarification:", len(filtered_candidates))

display(
    filtered_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Initial candidates: 2
Candidates after clarification: 1


,date,home_team,away_team,home_score,away_score,tournament,city,country
41204,2017-11-14,India,Myanmar,2,2,AFC Asian Cup qualification,Margao,India


In [18]:
# Cell 17 - Generate an answer from one selected result row

def generate_result_answer(intent, row):
    home_team = row["home_team"]
    away_team = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]
    date = row["date"]
    tournament = row["tournament"]
    city = row["city"]
    country = row["country"]
    neutral = row["neutral"]

    if intent == "home_team_score":
        return f"{home_team} scored {home_score} goal(s)."

    elif intent == "away_team_score":
        return f"{away_team} scored {away_score} goal(s)."

    elif intent == "match_score":
        return f"{home_team} {home_score} - {away_score} {away_team}."

    elif intent == "match_winner":
        if home_score > away_score:
            return f"{home_team} won the match {home_score}-{away_score}."
        elif away_score > home_score:
            return f"{away_team} won the match {away_score}-{home_score}."
        else:
            return f"The match ended in a {home_score}-{away_score} draw."

    elif intent == "total_goals":
        total = home_score + away_score
        return f"A total of {total} goal(s) were scored."

    elif intent == "match_date":
        return f"The match was played on {date}."

    elif intent == "match_location":
        return f"The match was played in {city}, {country}."

    elif intent == "tournament":
        return f"The match was part of the {tournament}."

    elif intent == "neutral_status":
        neutral_text = "Yes" if bool(neutral) else "No"
        return f"{neutral_text}, the match {'was' if bool(neutral) else 'was not'} played at a neutral venue."

    return "Answer generation for this intent is not implemented yet."

In [19]:
# Cell 18 - Test result answer generation

question = "Who won between India and Myanmar in 2017?"

intent = predict_intent(question)

candidates = find_result_candidates(question)

print("Intent:", intent)
print("Initial candidates:", len(candidates))

# User clarification
reply = "India"

filtered_candidates = apply_clarification(
    candidates,
    reply
)

print("Candidates after clarification:", len(filtered_candidates))

if len(filtered_candidates) == 1:
    selected_match = filtered_candidates.iloc[0]
    answer = generate_result_answer(intent, selected_match)
    print("Answer:", answer)
else:
    print(generate_clarification_question(filtered_candidates))

Intent: match_winner
Initial candidates: 2
Candidates after clarification: 1
Answer: The match ended in a 2-2 draw.


In [23]:
# Cell 19 - Interactive Results QA conversation

RESULT_INTENTS = {
    "home_team_score",
    "away_team_score",
    "match_date",
    "match_location",
    "match_score",
    "match_winner",
    "neutral_status",
    "total_goals",
    "tournament"
}


def ask_results_question(question):
    # Step 1: Predict intent
    intent = predict_intent(question)

    print(f"\nPredicted intent: {intent}")

    # Make sure this function is only handling results-based intents
    if intent not in RESULT_INTENTS:
        print(
            "This question belongs to another data source "
            "and will be handled in a later stage."
        )
        return

    # Step 2: Find initial candidate matches
    candidates = find_result_candidates(question)

    print(f"Matching records found: {len(candidates)}")

    # Nothing found
    if len(candidates) == 0:
        print(
            "\nAssistant: I couldn't find a match that fits "
            "the information in your question."
        )
        return

    # Step 3: Resolve ambiguity conversationally
    while len(candidates) > 1:

        clarification = generate_smart_clarification(candidates)

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print("\nAssistant: Please provide a little more information.")
            continue

        new_candidates = apply_clarification(candidates, reply)

        # Clarification didn't narrow anything
        if len(new_candidates) == len(candidates):
            print(
                "\nAssistant: That didn't narrow the matches down. "
                "Let's try another detail."
            )
            continue

        # Clarification accidentally removed everything
        if len(new_candidates) == 0:
            print(
                "\nAssistant: I couldn't match that detail to the "
                "remaining games. Please try another detail."
            )
            continue

        candidates = new_candidates

        print(f"\nRemaining matches: {len(candidates)}")

    # Step 4: Exactly one match remains
    selected_match = candidates.iloc[0]

    answer = generate_result_answer(
        intent,
        selected_match
    )

    print(f"\nAssistant: {answer}")

In [21]:
# Cell 20 - First interactive chatbot test

question = input("Ask a football question: ")

ask_results_question(question)

Ask a football question: Who won between India and Myanmar in 2017?

Predicted intent: match_winner
Matching records found: 2

Assistant: I found multiple matching games. Do you remember which country the game was played in? Options: Myanmar, India

You: India

Remaining matches: 1

Assistant: The match ended in a 2-2 draw.


In [22]:
# Cell 21 - Smarter clarification generator

def generate_smart_clarification(candidates):
    if len(candidates) == 0:
        return "I couldn't find a matching game."

    if len(candidates) == 1:
        return None

    # Build useful derived fields
    temp = candidates.copy()
    temp["year"] = temp["date"].astype(str).str[:4]

    # 1. Prefer year if multiple years remain
    years = sorted(temp["year"].dropna().unique().tolist())

    if len(years) > 1:
        # Avoid dumping too many options
        if len(years) <= 8:
            return (
                "I found matches from multiple years. "
                "Do you remember the year? "
                f"Options: {', '.join(years)}"
            )

        return (
            f"I found matches across {len(years)} different years. "
            "Do you remember roughly which year it was?"
        )

    # 2. Tournament
    tournaments = sorted(
        temp["tournament"].dropna().astype(str).unique().tolist()
    )

    if len(tournaments) > 1:
        if len(tournaments) <= 8:
            return (
                "I found more than one possible match. "
                "Do you remember the tournament? "
                f"Options: {', '.join(tournaments)}"
            )

        return (
            "I found matches from several tournaments. "
            "Do you remember which competition it was?"
        )

    # 3. Country
    countries = sorted(
        temp["country"].dropna().astype(str).unique().tolist()
    )

    if len(countries) > 1:
        return (
            "Do you remember which country the match was played in? "
            f"Options: {', '.join(countries)}"
        )

    # 4. City
    cities = sorted(
        temp["city"].dropna().astype(str).unique().tolist()
    )

    if len(cities) > 1:
        return (
            "Do you remember the city where the match was played? "
            f"Options: {', '.join(cities)}"
        )

    # 5. Home team
    home_teams = sorted(
        temp["home_team"].dropna().astype(str).unique().tolist()
    )

    if len(home_teams) > 1:
        return (
            "Do you remember which team was listed as the home team? "
            f"Options: {', '.join(home_teams)}"
        )

    # 6. Exact date only as a last resort
    dates = sorted(
        temp["date"].dropna().astype(str).unique().tolist()
    )

    if len(dates) > 1:
        return (
            "I still found more than one possible match. "
            "Do you remember the exact date? "
            f"Options: {', '.join(dates)}"
        )

    return (
        "I still found multiple matching records. "
        "Can you give me one more detail about the match?"
    )

In [24]:
question = "Who won between Brazil and Argentina?"
ask_results_question(question)


Predicted intent: match_winner
Matching records found: 110

Assistant: I found matches across 64 different years. Do you remember roughly which year it was?

You: 2021

Remaining matches: 2

Assistant: I found more than one possible match. Do you remember the tournament? Options: Copa América, FIFA World Cup qualification

You: World cup

Assistant: That didn't narrow the matches down. Let's try another detail.

Assistant: I found more than one possible match. Do you remember the tournament? Options: Copa América, FIFA World Cup qualification

You: FIFA World Cup

Assistant: That didn't narrow the matches down. Let's try another detail.

Assistant: I found more than one possible match. Do you remember the tournament? Options: Copa América, FIFA World Cup qualification

You: FIFA world cup qualification

Remaining matches: 1

Assistant: The match ended in a 0-0 draw.
